In [52]:
import requests
import pandas as pd
import time
import os
from fp.fp import FreeProxy
from tqdm import tqdm

In [ ]:
OUTPUT_DIR      = 'output'
URL_VACANCIES   = 'https://api.hh.ru/vacancies'
QUERIES         = [
    '\"Тестировщик\"', 
    '\"Аналитик данных\"', 
    '\"Продуктовый аналитик\"', 
    '\"Бизнес аналитик\"'
]

In [54]:
def get_vacancies(text: str, area: int = 1, pages: int = 2, per_page: int = 100):
    '''
    Возвращает (pages x per_page) строк информации по вакансиям \n

    text - описание вакансии                    \n
    area - регион (1 - Москва)                  \n
    pages - кол-во страниц                      \n
    per_page - кол-во возвращаемых вакансий     \n
    '''
    vacancies = []
    for page in range(pages):
        params = {'text': text, 'area': area, 'page': page, 'per_page': 100}
        response = requests.get(URL_VACANCIES, params=params)
        if response.status_code == 200:
            data = response.json()
            vacancies.extend(data['items'])
        time.sleep(0.25)  # Пауза, чтобы не нагружать API
    return vacancies

In [55]:
def get_proxy(country_id: str = 'RU', rand: bool = True, https: bool = False):
    ''' 
    Получение Proxy для смены промежуточного IP
    '''
    proxy = FreeProxy(country_id=[country_id], rand=rand, https=https).get()
    return proxy

In [56]:
def get_vacancy_card_info(vacancy_url: str, retry: int = 5):
    '''
    Запрос на информацию с карточки вакансии
    '''
    vacancy_info = None
    proxies = {}

    for i in range(1, retry + 1):
        # print(f"Попытка подключения {i}/{retry}")
        try:
            response = requests.get(vacancy_url, proxies=proxies)

            if response.status_code == 200:
                vacancy_info = response.json()
                break 
            raise ValueError("Некорректный код сервера")
        except Exception as e:
            # print(f"Смена proxy: {e}")
            proxies['http'] = get_proxy()

    return vacancy_info


In [57]:
def get_vacancy_info(vacancy_url: str):
    '''
    Возвращает информацию вакансии
    '''
    response = get_vacancy_card_info(vacancy_url)
    
    return response

In [58]:
vacancies = {}
area = 2 # СПб
count_executors = 10
# proxy = get_proxy()

def parse_vacancies():
    for query in QUERIES:
        vacancies_data = get_vacancies(query, area=area, per_page=5)

        vacancies[query] = []

        for vacancy_data in tqdm(vacancies_data, desc=query):
            vacancy_url = vacancy_data.get("url")

            if vacancy_url:
                vacancy_info = get_vacancy_info(vacancy_url)
                vacancies[query].append(vacancy_info)
    

parse_vacancies()

"Бизнес аналитик": 100%|██████████| 200/200 [02:01<00:00,  1.65it/s]


# Работа с отображением

In [59]:
def transform_vacancy_info(vacancy_info: dict) -> dict:
    row = {
        "Вакансия": vacancy_info.get("name"),
        "Вакансия, ссылка": vacancy_info.get("alternate_url"),
        "Локация": (vacancy_info.get("area") or {}).get("name"),
        "Зарплата, от": (vacancy_info.get("salary") or {}).get("from"),
        "Зарплата, до": (vacancy_info.get("salary") or {}).get("to"),
        "Зарплата, ео": (vacancy_info.get("salary") or {}).get("currency"),
        "Город": (vacancy_info.get("address") or {}).get("city"),
        "Опыт": (vacancy_info.get("experience") or {}).get("name"),
        "Занятость":(vacancy_info.get("employment") or {}).get("name"),
        "Работодатель": (vacancy_info.get("employer") or {}).get("name"),
        "Работодатель, ссылка": (vacancy_info.get("employer") or {}).get("alternate_url"),
        "Навыки": [x.get("name") for x in vacancy_info.get("key_skills")],
        "Формат работы": [x.get("name") for x in vacancy_info.get("work_format")],
    }
    return row


In [69]:
rows = []

for query, vacancies_list in vacancies.items():
    for vacancy_info in vacancies_list:
        row = {"Запрос": query}
        vacancy_info_t = transform_vacancy_info(vacancy_info)
        row.update(vacancy_info_t)
        rows.append(row)

df_total = pd.DataFrame(rows)
df_total.to_excel(os.path.join(OUTPUT_DIR, "total.xlsx"), index=False)
df_total

,Запрос,Вакансия,"Вакансия, ссылка",Локация,"Зарплата, от","Зарплата, до","Зарплата, ео",Город,Опыт,Занятость,Работодатель,"Работодатель, ссылка",Навыки,Формат работы
0,"""Тестировщик""",Тестировщик,https://hh.ru/vacancy/130293542,Санкт-Петербург,50000.0,NaN,RUR,Санкт-Петербург,Нет опыта,Полная занятость,Hotels.ru,https://hh.ru/employer/955808,"[SQL, Нагрузочное тестирование, Git, Usability...","[На месте работодателя, Удалённо, Гибрид]"
1,"""Тестировщик""",QA engineer,https://hh.ru/vacancy/130630758,Санкт-Петербург,NaN,NaN,NaN,NaN,От 1 года до 3 лет,Полная занятость,Суперлига,https://hh.ru/employer/12589064,"[Регрессионное тестирование, Функциональное те...",[Удалённо]
2,"""Тестировщик""",Инженер по тестированию,https://hh.ru/vacancy/130540570,Санкт-Петербург,NaN,NaN,NaN,Санкт-Петербург,Нет опыта,Полная занятость,СОЛВО,https://hh.ru/employer/537,"[SQL, Тестирование, Регрессионное тестирование]",[Гибрид]
3,"""Тестировщик""",Специалист отдела технической поддержки,https://hh.ru/vacancy/130623926,Санкт-Петербург,60000.0,NaN,RUR,NaN,Нет опыта,Полная занятость,Клиентикс,https://hh.ru/employer/1113074,"[Ведение переговоров, Грамотная речь, CRM, Гра...",[Удалённо]
4,"""Тестировщик""",QA Инженер,https://hh.ru/vacancy/130609560,Санкт-Петербург,150000.0,NaN,RUR,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,ЭБС Лань,https://hh.ru/employer/4027160,[],"[На месте работодателя, Удалённо]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,"""Бизнес аналитик""","Data Scientist в RecSys, middle+",https://hh.ru/vacancy/130110629,Санкт-Петербург,NaN,NaN,NaN,NaN,От 3 до 6 лет,Полная занятость,Контур,https://hh.ru/employer/41862,"[SASRec, CatBoost, CTR, collaborative filtering]",[Удалённо]
646,"""Бизнес аналитик""",Бизнес/Системный аналитик (1С-Битрикс),https://hh.ru/vacancy/130220926,Санкт-Петербург,190000.0,NaN,RUR,NaN,От 3 до 6 лет,Полная занятость,Ай-Пласт,https://hh.ru/employer/2463659,"[Бизнес-анализ, Системный анализ, BPMN, BABOK,...",[Удалённо]
647,"""Бизнес аналитик""",Системный аналитик,https://hh.ru/vacancy/129994961,Санкт-Петербург,70000.0,90000.0,RUR,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,"Северная Верфь, судостроительный завод",https://hh.ru/employer/61020,"[бизнес-процессы, Управление проектами, Бизнес...",[На месте работодателя]
648,"""Бизнес аналитик""",Бизнес-аналитик 1С,https://hh.ru/vacancy/130247943,Санкт-Петербург,100000.0,150000.0,RUR,Санкт-Петербург,От 1 года до 3 лет,Полная занятость,Авто-Стиль,https://hh.ru/employer/171028,[],[На месте работодателя]


In [70]:
rows_skills = []
for row in rows:
    for skill in row.get("Навыки"):
        row_skills = {
            "Запрос": row.get("Запрос"),
            "Навык": skill,
        }
        rows_skills.append(row_skills)
# 
df_skills = pd.DataFrame(rows_skills)
df_skills_agg = df_skills.groupby(by=["Запрос", "Навык"]).size().reset_index(name="Количество")
df_skills_sort = df_skills_agg.sort_values(by=["Запрос", "Количество"], ascending=[True, False])
df_skills_sort.to_excel(os.path.join(OUTPUT_DIR, "skills.xlsx"), index=False)
df_skills_sort

,Запрос,Навык,Количество
109,"""Аналитик данных""",SQL,23
147,"""Аналитик данных""",Анализ данных,23
166,"""Аналитик данных""",Аналитическое мышление,23
306,"""Аналитик данных""",Работа с большим объемом информации,19
99,"""Аналитик данных""",Python,18
...,...,...,...
1375,"""Тестировщик""",теория тестирования,1
1376,"""Тестировщик""",термодинамика,1
1377,"""Тестировщик""",тестирование ios,1
1378,"""Тестировщик""",тестирование оборудования,1
